<a href="https://colab.research.google.com/github/nadayali710-spec/Programming-For-AI-Project/blob/main/Hockey_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hockey Team Performance Analysis and Prediction

# Section 1: Web Scraping
**Goal:** scrape every hockey team-season row from the website into a pandas DataFrame (~580 rows, 9 columns), save the raw data with an untouched backup and metadata, then create a working copy with ~5% artificial missing values.

**Source:** the Scrape This Site "Hockey Teams: Forms, Searching and Pagination" practice pages (`https://www.scrapethissite.com/pages/forms/`).

**Outputs:** `hockey_raw.csv`, `hockey_raw_backup.csv`, `scrape_metadata.txt`, `hockey_working_missing.csv`

## Step 1 - Fetch and inspect the HTML

### Cell 1 - Imports
We use `requests` to download pages, `BeautifulSoup` to parse HTML, and `pandas` to build tables.

In [2]:
import time
import datetime as dt
import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

print("requests", requests.__version__)
print("pandas", pd.__version__)

requests 2.32.4
pandas 2.2.3


### Cell 2 - Fetch one page
We request the first page and check the HTTP status code (`200` means success). The site paginates with the `page_num` query parameter and we ask for 100 rows per page.

In [3]:
BASE_URL = "https://www.scrapethissite.com/pages/forms/"
HEADERS = {"User-Agent": "Mozilla/5.0 (student project - PFAI hockey analysis)"}

resp = requests.get(BASE_URL, params={"page_num": 1, "per_page": 100}, headers=HEADERS, timeout=30)
print("Status code:", resp.status_code)
print("Final URL  :", resp.url)
print("Length     :", len(resp.text), "characters")

Status code: 200
Final URL  : https://www.scrapethissite.com/pages/forms/?page_num=1&per_page=100
Length     : 135648 characters


### Cell 3 - Look at the raw HTML
Before writing a parser we must *see* the structure of the page.

In [4]:
print(resp.text[:1500])

<!doctype html>
<html lang="en">
  <head>
    <meta charset="utf-8">
    <title>Hockey Teams: Forms, Searching and Pagination | Scrape This Site | A public sandbox for learning web scraping</title>
    <link rel="icon" type="image/png" href="/static/images/scraper-icon.png" />

    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <meta name="description" content="Browse through a database of NHL team stats since 1990. Practice building a scraper that handles common website interface components.">

    <link href="https://maxcdn.bootstrapcdn.com/bootstrap/3.3.5/css/bootstrap.min.css" rel="stylesheet" integrity="sha256-MfvZlkHCEqatNoGiOXveE8FIwMzZg4W85qfrfIFBfYc= sha512-dTfge/zgoMYpP7QbHy4gWMEGsbsdZeCXz7irItjcC3sPUFtf0kuFbDz/ixG7ArTxmDjLXDmezHubeNikyKGVyQ==" crossorigin="anonymous">
    <link href='https://fonts.googleapis.com/css?family=Lato:400,700' rel='stylesheet' type='text/css'>
    <link rel="stylesheet" type="text/css" href="/static/css/styles.css">

   

### Cell 4 - Parse with BeautifulSoup and find the table
`BeautifulSoup` turns the HTML text into a tree we can search. The data lives in a `<table class="table">`.

In [5]:
soup = BeautifulSoup(resp.text, "html.parser")
table = soup.find("table", class_="table")
print("Table found:", table is not None)

header_cells = [th.get_text(strip=True) for th in table.find_all("th")]
print("Column headers:", header_cells)

Table found: True
Column headers: ['Team Name', 'Year', 'Wins', 'Losses', 'OT Losses', 'Win %', 'Goals For (GF)', 'Goals Against (GA)', '+ / -']


### Cell 5 - Find the data rows
Each team-season is a `<tr class="team">` row. We count them and print the first one.

In [6]:
rows = table.find_all("tr", class_="team")
print("Rows on this page:", len(rows))
print(rows[0].prettify()[:900])

Rows on this page: 100
<tr class="team">
 <td class="name">
  Boston Bruins
 </td>
 <td class="year">
  1990
 </td>
 <td class="wins">
  44
 </td>
 <td class="losses">
  24
 </td>
 <td class="ot-losses">
 </td>
 <td class="pct text-success">
  0.55
 </td>
 <td class="gf">
  299
 </td>
 <td class="ga">
  264
 </td>
 <td class="diff text-success">
  35
 </td>
</tr>



### Cell 6 - Inspect the cells inside one row
Each `<td>` has a class name (`name`, `year`, `wins`, ...). Those class names give us reliable column names.

In [7]:
first = rows[0]
for td in first.find_all("td"):
    print(f"{td.get('class')[0]:<10} -> {td.get_text(strip=True)!r}")

name       -> 'Boston Bruins'
year       -> '1990'
wins       -> '44'
losses     -> '24'
ot-losses  -> ''
pct        -> '0.55'
gf         -> '299'
ga         -> '264'
diff       -> '35'


## Step 2 - Parse rows and scrape all pages

### Cell 7 - Row parser
One function turns one `<tr>` into a Python dictionary. Column names are tidied (`OT Losses` becomes `OT_Losses`, `Win %` becomes `Win_Pct`, `+ / -` becomes `Diff`).

In [8]:
CLASS_TO_COL = {
    "name": "Team", "year": "Year", "wins": "Wins", "losses": "Losses",
    "ot-losses": "OT_Losses", "pct": "Win_Pct", "gf": "GF", "ga": "GA", "diff": "Diff",
}

def parse_row(tr):
    """Convert one <tr class='team'> into a dict {column: text}."""
    record = {}
    for td in tr.find_all("td"):
        key = td.get("class")[0]
        record[CLASS_TO_COL[key]] = td.get_text(strip=True)
    return record

parse_row(rows[0])

{'Team': 'Boston Bruins',
 'Year': '1990',
 'Wins': '44',
 'Losses': '24',
 'OT_Losses': '',
 'Win_Pct': '0.55',
 'GF': '299',
 'GA': '264',
 'Diff': '35'}

### Cell 8 - `scrape_page` function
Download one page and return a list of row dictionaries. A short `sleep` keeps us polite to the server.

In [ ]:
def scrape_page(page_num, per_page=100):
    r = requests.get(BASE_URL, params={"page_num": page_num, "per_page": per_page},
                     headers=HEADERS, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    trs = soup.find_all("tr", class_="team")
    time.sleep(0.5)
    return [parse_row(tr) for tr in trs]

test = scrape_page(1)
print(len(test), "rows from page 1")
test[:2]